## Introduction


Create spark session and import spark_data_check module

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('my_app').config("spark.sql.ansi.enabled", "false").getOrCreate()

from pyspark.sql import functions as F
from spark_data_check import SparkDataCheck

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 17:21:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/26 17:21:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/26 17:21:29 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/26 17:21:29 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/03/26 17:21:29 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


Use read_csv() method to create an instance of the SparkDataCheck class

In [2]:
air = SparkDataCheck.read_csv(spark, file_path = 'air.csv')

# drop extra index column
air.df = air.df.drop("_c0")

# replace periods in column names with underscores to avoid errors
new_cols = [c.replace(".", "_") for c in air.df.columns]
air.df = air.df.toDF(*new_cols)

# show first 5 rows of data frame
air.df.show(5)

+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|3/10/2004|2026-03-26 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|3/10/2004|2026-03-26 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|3/10/2004|2026-03-26 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|3/10/2004|2026-03-26 21:00:00|   2.2|       1376|      80|     9.2|        

Print air data schema to look at column names and types

In [34]:
air.df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08.S1(CO): integer (nullable = true)
 |-- NMHC(GT): integer (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08.S2(NMHC): integer (nullable = true)
 |-- NOx(GT): integer (nullable = true)
 |-- PT08.S3(NOx): integer (nullable = true)
 |-- NO2(GT): integer (nullable = true)
 |-- PT08.S4(NO2): integer (nullable = true)
 |-- PT08.S5(O3): integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)



Test the methods in the SparkDataCheck class using the air data

### Method: check_numeric_col()

For the first method, check_numeric_col(), we will first test it by providing 'CO(GT)' as the numeric column.

The function has worked as intended in this case, we can see all of the values were correctly assigned 'true' or 'false' based on whether they are between the provided bounds. 

In [14]:
air.check_numeric_col(column = 'CO(GT)', lower = 0, upper = 10)
air.df.select('CO(GT)', 'within_num_range').show()

+------+----------------+
|CO(GT)|within_num_range|
+------+----------------+
|   2.6|            true|
|   2.0|            true|
|   2.2|            true|
|   2.2|            true|
|   1.6|            true|
|   1.2|            true|
|   1.2|            true|
|   1.0|            true|
|   0.9|            true|
|   0.6|            true|
|-200.0|           false|
|   0.7|            true|
|   0.7|            true|
|   1.1|            true|
|   2.0|            true|
|   2.2|            true|
|   1.7|            true|
|   1.5|            true|
|   1.6|            true|
|   1.9|            true|
+------+----------------+
only showing top 20 rows


Now we will test that the error messages work properly.

First, try providing no bounds. We will test the method on the column 'T' this time.

The result is as intended, with an error message being printed saying to provide at least one bound. 

In [15]:
air.check_numeric_col(column = 'T')

Error: Must provide at least one of 'upper' or 'lower'


Now we will test the method by only providing one bound.
This test worked as expected, with the column only being compared to the provided lower bound. 

In [20]:
air.check_numeric_col(column = 'T', lower = 10)
air.df.select('T', 'within_num_range').show()

+----+----------------+
|   T|within_num_range|
+----+----------------+
|13.6|            true|
|13.3|            true|
|11.9|            true|
|11.0|            true|
|11.2|            true|
|11.2|            true|
|11.3|            true|
|10.7|            true|
|10.7|            true|
|10.3|            true|
|10.1|            true|
|11.0|            true|
|10.5|            true|
|10.2|            true|
|10.8|            true|
|10.5|            true|
|10.8|            true|
|10.5|            true|
| 9.5|           false|
| 8.3|           false|
+----+----------------+
only showing top 20 rows


Lastly, we will test the method by providing a non-numeric column. The 'Time' column is a timestamp, so we will use it for this test.

The result is as expected, with an error message printing telling us to provide a numeric column.

In [12]:
air.check_numeric_col(column = 'Time', lower = 0, upper = 10)

Error: Column must of type float, int, longint, bigint, double, or integer


### Method: check_string_col

Now we will test the method check_string_col(). This method accepts a column name and a set of levels and returns a data frame with a column added indicating whether the values of the column are within the user specified levels. 

First, we will test the method using the 'Date' column, which is a string.

The result is as expected - a boolean column called 'within_level' was added and we can see that the values of within_level are correct.

In [17]:
air.check_string_col(column = 'Date', levels = ['3/10/2004', '3/11/2004'])
air.df.select('Date', 'within_level').show()

+---------+------------+
|     Date|within_level|
+---------+------------+
|3/10/2004|        true|
|3/10/2004|        true|
|3/10/2004|        true|
|3/10/2004|        true|
|3/10/2004|        true|
|3/10/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
+---------+------------+
only showing top 20 rows


Now let's see what we get when we provide levels that are not in the provided column.

All of the 'within_level' values are false, so the method worked correctly.

In [18]:
air.check_string_col(column = 'Date', levels = ['0', '1', '2'])
air.df.select('Date', 'within_level').show()

+---------+------------+
|     Date|within_level|
+---------+------------+
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
+---------+------------+
only showing top 20 rows


Next we can test the method by providing a level that is in the Date column and a level that is not in the date column.

All of the rows where Date = '3/10/2004' are true for the within_level column and all of the other rows are false for within_level, so this worked correctly.

In [19]:
air.check_string_col(column = 'Date', levels = ['3/10/2004', '1'])
air.df.select('Date', 'within_level').show()

+---------+------------+
|     Date|within_level|
+---------+------------+
|3/10/2004|        true|
|3/10/2004|        true|
|3/10/2004|        true|
|3/10/2004|        true|
|3/10/2004|        true|
|3/10/2004|        true|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
+---------+------------+
only showing top 20 rows


Lastly, we will test the method by providing a non-string column to the function.

This correctly returned an error message stating that the provided column must be a string column.

In [9]:
air.check_string_col(column = 'T', levels = [1, 2, 3])

Error: Column type must be string.


### Method: check_missing_values()

Next, we wil test the check_missing_values() method. This method checks a column for missing values and returns the modified data frame with a boolean column indicating whether the values in a column are missing. 

Let's first test the function on a numeric column. 

The 'is_missing' column was added properly and correctly show that none of these values of 'AH' are missing. 

In [ ]:
air.check_missing_values(column = 'AH')
air.df.select('AH', 'is_missing').show()

Let's now test this method on a different type of column. We will test it on the 'Time' column, which is a timestamp.

Again, the result is as expected- none of the values are missing and all of the 'is_missing' values say false.

In [ ]:
air.check_missing_values(column = 'Time')
air.df.select('Time', 'is_missing').show()

In order to confirm this method works as intended, we want to test it on a column that has null values. Let's check the number of null values in each column to see which column we can test this method on.

In [ ]:
air.df.select([
    F.sum(F.when(F.col(f"`{c}`").isNull(), 1).otherwise(0)).alias(c)
    for c in air.df.columns
]).show()

Currently, none of the columns have null values, so we can't confirm that the method correctly identifies null values. 

However, we know from the first project that the variables 'C6H6(GT)' and 'CO(GT)' contains missing values, which are coded as '-200'. We can reassign these values as nulls and then test our check_missing_values() method. 

Then we can rerun the code to check the number of missing values per column to make sure this worked.

CO(GT) now has 1683 missing values and C6H6(GT) has 366 missing values, so we can test our method on these columns.

In [ ]:
# columns to reassign values to missing for
cols = ['CO(GT)', 'C6H6(GT)']

# replace values for both columns
for col in cols:
    air.df = air.df.withColumn(
        col,
        F.when(F.col(col) == -200, None).otherwise(F.col(col))
    )

# get null value counts after replacing values
air.df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in air.df.columns
]).show()

Now we can test the method on these two columns. 

In this output, there is one null value for CO(GT), and we can see that check_missing_values correctly assigned 'false' to this row.

In [ ]:
air.check_missing_values(column = 'CO(GT)')
air.df.select('CO(GT)', 'is_missing').show()

Now let's test the method on C6H6(GT).

This time, we will return the count of true and false values from the 'is_missing' column to make sure the number of true values aligns with the number of nulls in 'C6H6(GT)'. 

This returned 366, which is the same number of null values in the column.

In [ ]:
air.check_missing_values(column = 'C6H6(GT)')
air.df.select('C6H6(GT)', 'is_missing').groupby('is_missing').count().show()

### Method: summarize_min_max()

This method takes in a numeric column and reports the minimum and maximum values of the column. A grouping column can optionally be provided. If no column name is provided, all numeric columns will be summarized.

First, let's test the method using a numeric column with no grouping column.

The output is a pandas data frame with 'CO(GT)_min' and 'CO(GT)_max' columns, as expected.

In [7]:
air.summarize_min_max(column = 'CO(GT)')

,CO(GT)_min,CO(GT)_max
0,0.1,11.9


Now lets test the method using 'Date' as the grouping column and 'C6H6(GT)' as the numeric column to summarize.

I used sort_values() to sort the result by the Date column to make it easier to read.

This returned the minimum and maximum values of 'C6H6(GT)' for each value of 'Date' as expected.

In [8]:
air.summarize_min_max(column = 'C6H6(GT)', group_col = 'Date').sort_values('Date')

,Date,C6H6(GT)_max,C6H6(GT)_max
350,1/1/2005,3.0,16.6
308,1/10/2005,1.3,21.4
117,1/11/2005,2.1,26.5
36,1/12/2005,4.2,28.4
253,1/13/2005,3.2,26.6
...,...,...,...
172,9/5/2004,1.9,30.3
39,9/6/2004,1.0,13.5
382,9/7/2004,2.7,22.0
131,9/8/2004,10.6,34.5


Now let's try passing no numeric column and no grouping column to the method. It should find all numeric columns in the data frame and find the minimum and maximum values for each one.

In [ ]:
air.summarize_min_max()

Next, we can try passing a column to group by, but no numeric column. The method should find all numeric columns in the data frame and summarize the minimum and maximum values of each, grouped by the given grouping column. We will again use 'Date' as the grouping column.

In [ ]:
air.summarize_min_max(group_col = 'Date').sort_values('Date')